# Module 40 — Customer Lifetime Value: Gamma-Gamma & BG/NBD Models

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: Lifetimes, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 38 (RFM Segmentation)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Transaction Data](#3-synthetic-transaction-data)
4. [BG/NBD Model](#4-bgnbd-model)
5. [Gamma-Gamma Model](#5-gamma-gamma-model)
6. [CLV Prediction](#6-clv-prediction)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why Customer Lifetime Value (CLV)?

CLV predicts **total revenue** from a customer:
- **Acquisition ROI**: How much to spend acquiring customers.
- **Retention budget**: How much to invest in retention.
- **Segmentation**: Identify high-value customers.

**Applications**:
1. **Marketing budget**: Allocate spend based on expected CLV.
2. **Personalization**: Tailor experiences by CLV segment.
3. **Churn prevention**: Prioritize high-CLV at-risk customers.

**Key models**:
- **BG/NBD**: Predict future purchases (frequency).
- **Gamma-Gamma**: Predict monetary value per transaction.
- **CLV = Frequency × Monetary × Margin**.

In this notebook we will:
1. Generate synthetic transaction data.
2. Fit BG/NBD model for purchase frequency.
3. Fit Gamma-Gamma model for monetary value.
4. Predict 12-month CLV for each customer.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── CLV libraries ────────────────────────────────────────────────
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_log

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Transaction Data

In [ ]:
# Generate synthetic transaction data
N_CUSTOMERS = 5000
N_TRANSACTIONS = 50000

# Generate transactions with realistic patterns
transactions = []
for customer_id in range(N_CUSTOMERS):
    # Customer characteristics
    purchase_rate = np.random.exponential(30)  # days between purchases
    avg_amount = np.random.exponential(50)
    
    # Generate transactions
    current_date = pd.Timestamp('2023-01-01') + pd.Timedelta(days=np.random.randint(0, 365))
    end_date = pd.Timestamp('2024-01-01')
    
    while current_date < end_date:
        amount = np.random.exponential(avg_amount)
        transactions.append({
            'customer_id': customer_id,
            'date': current_date,
            'amount': amount
        })
        current_date += pd.Timedelta(days=np.random.exponential(purchase_rate))

df_transactions = pd.DataFrame(transactions)

print(f"✓ Generated {len(df_transactions):,} transactions")
print(f"  Customers: {N_CUSTOMERS:,}")
print(f"  Date range: {df_transactions['date'].min()} to {df_transactions['date'].max()}")
print(f"  Total revenue: ${df_transactions['amount'].sum():,.2f}")

---
## §4 · BG/NBD Model

In [ ]:
# Prepare data for BG/NBD model
summary = summary_data_from_transaction_log(
    df_transactions,
    customer_id_col='customer_id',
    datetime_col='date',
    monetary_value_col='amount',
    observation_period_end='2024-01-01'
)

print(f"✓ Prepared summary data")
print(f"  Customers: {len(summary):,}")
print(f"\nSummary statistics:")
print(summary.describe())

# Fit BG/NBD model
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(summary['frequency'], summary['recency'], summary['T'])

print(f"\n✓ BG/NBD model fitted")
print(f"  Parameters: {bgf.params_}")

---
## §5 · Gamma-Gamma Model

In [ ]:
# Filter customers with at least one repeat purchase
summary_filtered = summary[summary['frequency'] > 0]

# Fit Gamma-Gamma model
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(
    summary_filtered['frequency'],
    summary_filtered['monetary_value']
)

print(f"✓ Gamma-Gamma model fitted")
print(f"  Parameters: {ggf.params_}")
print(f"  Customers with repeat purchases: {len(summary_filtered):,}")

---
## §6 · CLV Prediction

In [ ]:
# Predict 12-month CLV
summary['predicted_purchases'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    365,  # 12 months
    summary['frequency'],
    summary['recency'],
    summary['T']
)

# For customers with repeat purchases, use Gamma-Gamma
summary.loc[summary_filtered.index, 'predicted_clv'] = ggf.customer_lifetime_value(
    bgf,
    summary_filtered['frequency'],
    summary_filtered['recency'],
    summary_filtered['T'],
    summary_filtered['monetary_value'],
    time=12,  # months
    discount_rate=0.01  # monthly discount rate
)

# Fill NaN with 0 for customers without repeat purchases
summary['predicted_clv'] = summary['predicted_clv'].fillna(0)

print(f"✓ CLV predictions complete")
print(f"\nCLV Statistics:")
print(f"  Mean CLV: ${summary['predicted_clv'].mean():,.2f}")
print(f"  Median CLV: ${summary['predicted_clv'].median():,.2f}")
print(f"  Total CLV: ${summary['predicted_clv'].sum():,.2f}")
print(f"\nTop 10 customers by CLV:")
print(summary.nlargest(10, 'predicted_clv')[['predicted_clv']])

---
## §7 · Visualization

In [ ]:
# Visualize CLV distribution
fig = make_subplots(rows=1, cols=2, subplot_titles=['CLV Distribution', 'CLV vs Frequency'])

# CLV distribution
fig.add_trace(go.Histogram(
    x=summary['predicted_clv'],
    nbinsx=50,
    name='CLV Distribution'
), row=1, col=1)

# CLV vs Frequency
fig.add_trace(go.Scatter(
    x=summary['frequency'],
    y=summary['predicted_clv'],
    mode='markers',
    name='Customers',
    marker=dict(size=5, opacity=0.5)
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - CLV distribution is highly skewed")
print("   - Frequency is a strong predictor of CLV")
print("   - Focus retention on high-CLV customers")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Marketing Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Acquisition budgeting, retention targeting, personalization |
> | **Models** | BG/NBD (frequency) + Gamma-Gamma (monetary) |
> | **Prediction** | 12-month CLV for strategic planning |
> | **Evaluation** | Compare predicted vs actual on holdout |
> | **Deployment** | Monthly CLV scoring, real-time for high-value |
>
> ### 💡 CLV-Based Strategies
>
> | CLV Segment | Strategy | Budget Allocation |
> |-------------|----------|------------------|
> | High CLV (> $500) | Premium experiences, loyalty rewards | High |
> | Medium CLV ($100-500) | Targeted campaigns, upsell | Medium |
> | Low CLV (< $100) | Automated marketing, self-service | Low |
>
> ### 🔑 Key Takeaways
>
> 1. **CLV predicts total customer value** over their lifetime.
> 2. **BG/NBD + Gamma-Gamma** is the industry standard model.
> 3. **Frequency is the strongest predictor** of CLV.
> 4. **Use CLV for resource allocation** and targeting.
> 5. **Regular re-calibration** is essential as behavior changes.